In [1]:
%pip install neo4j openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # override=True ensures that .env is loaded even if the env vars are already set

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...VmEA


In [4]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # override=True ensures that .env is loaded even if the env vars are already set

token = os.environ.get("NEO4J_PASSWORD")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"NEO4J_PASSWORD loaded ({len(token)} characters): {masked}")
else:
    print("NEO4J_PASSWORD not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'NEO4J_PASSWORD', and load_dotenv() ran without error.")

NEO4J_PASSWORD loaded (43 characters): 8QGO...EuxY


In [5]:
"""
Retrieval-augmented question answering over your Neo4j knowledge graph.

Pulls every relationship touching any entity whose name contains TOPIC,
formats those rows (with chunk_id/doi/category so every claim is traceable
back to a source sentence), and sends them to OpenAI with instructions to
answer strictly from that context, nothing else. This is the thing that
actually answers "what does the literature say about X", not clicking
through the graph node by node.

Works against Aura or Desktop identically, it's just a network connection
either way -- point NEO4J_URI at whichever one you're using.

Designed for Jupyter/Colab execution. No __main__ guard -- just set the
CONFIG values below and run the whole cell/file.

pip install neo4j openai --break-system-packages
"""

import os

from neo4j import GraphDatabase
from openai import OpenAI

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
NEO4J_URI = "neo4j+s://f1cebb95.databases.neo4j.io"     # <- your Aura instance URI
NEO4J_USERNAME = "f1cebb95"
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")   # <- fill in, or set the env var
NEO4J_DATABASE = "f1cebb95"

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")   # <- fill in, or set the env var
OPENAI_MODEL = "gpt-4o"

TOPIC = "ultrasonication"   # <- substring match against entity names, case-insensitive.
                            #    Change this per question, e.g. "pH shifting", "gel strength".

# Safety cap on context size, in estimated tokens (roughly chars/4), not row
# count -- a flat row cap would either truncate a legitimately large topic
# or fail to protect a verbose one. 100,000 leaves headroom under GPT-4o's
# context window for the prompt itself and the response.
MAX_CONTEXT_TOKENS = 100_000

QUESTION = (
    "Write a detailed overview of {topic}'s effect on the physicochemical and "
    "structural properties of the plant protein sources represented in the data."
)

REQUIREMENTS = """\
Requirements:
- Base every claim strictly on patterns actually present in the filtered rows. Do not supplement with anything not in the data.
- Where the data allows, compare {topic}'s effects against other treatments, and note any rows where {topic} was used in combination with another treatment (not alone).
- Organize the overview around genuine patterns in the data (e.g. consistent effects across sources, effects that vary by protein source, or conflicting results) rather than a flat list of individual findings.
- Every claim must cite the specific chunk_id it comes from, e.g. 10-1007_s13197-020-04241-8_results_discussion1. Do not make a claim without a citation.
"""

OUT_PATH = "rag_answer.md"

# ---------------------------------------------------------------
# CONNECT + RETRIEVE
# ---------------------------------------------------------------
if not NEO4J_PASSWORD:
    raise SystemExit("Set NEO4J_PASSWORD above or as an environment variable first.")
if not OPENAI_API_KEY:
    raise SystemExit("Set OPENAI_API_KEY above or as an environment variable first.")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connected to {NEO4J_URI}")

# Undirected match so TOPIC hits the entity whether it's the source or the
# target; startNode/endNode (not n/m) recover the real triple direction
# regardless of which side matched. DISTINCT guards against a relationship
# being returned twice when both of its endpoints happen to match TOPIC.
# Node labels are pulled too, so "combined treatment" detection below can
# tell a real two-treatment combination (e.g. "X and ultrasonication",
# both ModificationMethod/ExtractionMethod) apart from a compound property
# name that merely contains the word "and" (e.g. "denaturation and
# unfolding", a StructuralProperty) -- a plain text search on "and" can't
# tell those apart, but the node's own label can.
CYPHER = """
MATCH (n)-[r]-(m)
WHERE toLower(n.name) CONTAINS toLower($topic)
RETURN DISTINCT
    startNode(r).name AS source,
    labels(startNode(r))[0] AS source_label,
    type(r) AS interaction_type,
    endNode(r).name AS target,
    labels(endNode(r))[0] AS target_label,
    r.category AS category,
    r.chunk_id AS chunk_id,
    r.doi AS doi,
    r.compared_property AS compared_property,
    r.reported_value AS reported_value,
    r.claim_status AS claim_status,
    r.corresponding_sentence AS corresponding_sentence
"""

with driver.session(database=NEO4J_DATABASE) as session:
    rows = [dict(record) for record in session.run(CYPHER, topic=TOPIC)]
driver.close()

print(f"Retrieved {len(rows)} rows matching '{TOPIC}'")
if not rows:
    raise SystemExit(f"No relationships found for entities containing '{TOPIC}'. Check spelling.")

# ---------------------------------------------------------------
# FORMAT CONTEXT -- one block per row, token-budgeted
# ---------------------------------------------------------------
TREATMENT_LABELS = {"ModificationMethod", "ExtractionMethod"}


def format_row(row):
    # Only flag as a combined treatment when the "and" is in an entity that
    # is itself a treatment/method node -- see the CYPHER comment above.
    source_is_combo = " and " in row["source"].lower() and row["source_label"] in TREATMENT_LABELS
    target_is_combo = " and " in row["target"].lower() and row["target_label"] in TREATMENT_LABELS
    combo_note = " [combined treatment]" if (source_is_combo or target_is_combo) else ""
    return (
        f"- {row['source']} --[{row['interaction_type']}]--> {row['target']}"
        f"{combo_note}\n"
        f"  category: {row['category']} | compared_property: {row['compared_property']} | "
        f"reported_value: {row['reported_value']} | claim_status: {row['claim_status']}\n"
        f"  chunk_id: {row['chunk_id']} | doi: {row['doi']}\n"
        f"  sentence: \"{row['corresponding_sentence']}\""
    )

blocks = [format_row(r) for r in rows]

kept, running_chars = [], 0
for b in blocks:
    running_chars += len(b)
    if running_chars // 4 > MAX_CONTEXT_TOKENS:
        break
    kept.append(b)

if len(kept) < len(blocks):
    print(
        f"WARNING: full context for '{TOPIC}' is ~{running_chars // 4} estimated tokens, "
        f"over the {MAX_CONTEXT_TOKENS} budget. Using the first {len(kept)} of {len(blocks)} "
        f"rows. Narrow TOPIC, raise MAX_CONTEXT_TOKENS, or run this multiple times over "
        f"sub-topics if you need full coverage -- don't treat a truncated answer as complete."
    )

context_block = "\n\n".join(kept)

# ---------------------------------------------------------------
# ASK THE LLM
# ---------------------------------------------------------------
question_text = QUESTION.format(topic=TOPIC)
requirements_text = REQUIREMENTS.format(topic=TOPIC)

system_prompt = (
    "You are analyzing a filtered extract from a plant-protein food-science knowledge "
    "graph built from published literature. Answer using ONLY the context rows provided "
    "in the user message. Never draw on outside knowledge, training data, or anything not "
    "explicitly present in that context, even if you believe it to be true. If the context "
    "doesn't support a claim, do not make it."
)

user_prompt = f"""{question_text}

{requirements_text}

Context ({len(kept)} rows):
{context_block}
"""

client = OpenAI(api_key=OPENAI_API_KEY)
response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0,
)

answer = response.choices[0].message.content
print("\n" + "=" * 70 + "\n")
print(answer)

with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write(f"# {question_text}\n\n{answer}\n")
print(f"\nSaved to {OUT_PATH}")

Connected to neo4j+s://f1cebb95.databases.neo4j.io
Retrieved 588 rows matching 'ultrasonication'


Ultrasonication has been shown to significantly impact the physicochemical and structural properties of plant proteins, with effects varying based on the frequency and combination with other treatments. Here's an overview based on the provided data:

### Effects of Ultrasonication Alone

1. **Physicochemical Properties:**
   - **Increased Solubility and Emulsification:** Ultrasonication generally increases solubility and emulsification activity across various plant proteins. For instance, it significantly improved the solubility of proteins like RPSPI and PMPI, with increases up to 32.1% (10-1016_j-ijbiomac-2025-143154_results_discussion1). Emulsification activity also increased, with notable improvements in emulsifying activity index (10-1016_j-foodchem-2024-138671_results_discussion2).
   - **Foaming and Water Holding Capacities:** Ultrasonication enhances foaming capacity and water hol